# DDPM tự viết trên FashionMNIST

Bạn ghép các mảnh của bài Basic 03 (schedule, $$q(x_t \mid x_0)$$, posterior) thành một diffusion model hoàn chỉnh:
một U-Net nhỏ có time embedding dự đoán nhiễu, loss "simple" của Ho et al. (2020), sampler DDPM tổ tiên 1 000 bước
và sampler DDIM ít bước, cộng EMA cho trọng số. Rồi đo: sampler nào, bao nhiêu bước, schedule nào, có EMA hay không
thì mẫu sinh tốt hơn, theo một classifier FashionMNIST làm giám khảo.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`. Không dùng `diffusers` trong cell chấm điểm (một cell
  `explore` cuối bài đối chiếu với nó).
- Cell gắn tag `explore` đọc dữ liệu thật, train, vẽ; test bỏ qua chúng. Cell đầu tiên tải FashionMNIST (30 MB) vào
  `data/raw/fashion-mnist` nếu chưa có, bằng `torchvision`.
- Hai cấu hình chọn tự động. Env gốc (CPU): train 600 bước, cả notebook khoảng 10 phút; đủ để kiểm code chạy đúng,
  mẫu còn mờ. Kernel `ai-gpu` (GPU 4GB): 8 000 bước cho mỗi schedule, cả notebook khoảng 20 phút; các số trong lời giải mẫu đo ở cấu hình này.
- Chấm: `uv run pytest tracks/ai-engineer/modules/06-generative-models/exercises/intermediate/03-ddpm-fashion-mnist -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải FashionMNIST bằng torchvision nếu chưa có (30 MB, chỉ lần đầu)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FASHION = ROOT / "data/raw/fashion-mnist"
if not (FASHION / "FashionMNIST/raw/t10k-labels-idx1-ubyte.gz").exists():
    import torchvision

    for train in (True, False):
        torchvision.datasets.FashionMNIST(FASHION, train=train, download=True)
print(sorted(p.name for p in (FASHION / "FashionMNIST/raw").iterdir()))

In [ ]:
import gzip
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


def load_idx_images(root: Path, split: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Tệp idx gz (cùng định dạng MNIST) → (X float32 (N, 1, 28, 28) trong [-1, 1], y int64). split: "train" | "t10k"."""
    def read_idx(path: Path) -> np.ndarray:
        data = gzip.open(path).read()
        ndim = data[3]
        shape = np.frombuffer(data, ">u4", ndim, offset=4).astype(int)
        return np.frombuffer(data, np.uint8, offset=4 + 4 * ndim).reshape(shape)

    X = read_idx(Path(root) / f"{split}-images-idx3-ubyte.gz").astype(np.float32) / 127.5 - 1
    y = read_idx(Path(root) / f"{split}-labels-idx1-ubyte.gz").astype(np.int64)
    return torch.from_numpy(X).unsqueeze(1), torch.from_numpy(y)

## 1. Schedule

`make_schedule(T=1000, kind="linear")` trả dict ba tensor **float32** shape `(T,)`: `betas`, `alphas` ($$1 - \beta$$) và
`alpha_bars` ($$\bar\alpha_t = \prod_{s \le t}\alpha_s$$). `kind="linear"`: $$\beta$$ từ `1e-4` tới `0.02`; `kind="cosine"`: schedule
cosine với $$s = 0.008$$, $$\beta$$ chặn ở 0.999 (công thức ở bài Basic 03, mục 5 lý thuyết). Tính bằng float64 rồi mới đổi sang
float32. Quy ước chỉ số như Basic 03: `t = 0..T-1`.

In [ ]:
def make_schedule(T: int = 1000, kind: str = "linear") -> dict[str, torch.Tensor]:
    """{"betas", "alphas", "alpha_bars"}, float32 shape (T,). kind: "linear" | "cosine"."""
    # TODO
    raise NotImplementedError

## 2. Mạng dự đoán nhiễu

- `sinusoidal_embedding(t, dim)`: `t` shape `(B,)` (số nguyên), trả `(B, dim)` float32 với `dim` chẵn, $$h = \text{dim}/2$$,
  tần số $$\omega_i = 10000^{-i/h}$$ ($$i = 0..h-1$$); nửa đầu là $$\sin(t\,\omega_i)$$, nửa sau $$\cos(t\,\omega_i)$$ (như position
  encoding của Transformer).
- `ResBlock(cin, cout, tdim)`: thuộc tính `norm1 = GroupNorm(8, cin)`, `conv1 = Conv2d(cin, cout, 3, padding=1)`,
  `temb = Linear(tdim, cout)`, `norm2 = GroupNorm(8, cout)`, `conv2 = Conv2d(cout, cout, 3, padding=1)`, `skip`
  (`Conv2d(cin, cout, 1)` khi `cin != cout`, ngược lại `nn.Identity()`). `forward(x, emb)`:
  $$h = \text{conv1}(\text{SiLU}(\text{norm1}(x))) + \text{temb}(\text{emb})$$ (cộng theo kênh, broadcast mọi vị trí), rồi
  $$\text{out} = \text{conv2}(\text{SiLU}(\text{norm2}(h))) + \text{skip}(x)$$.

Cell sau (không chấm) ghép chúng thành `UNet(ch=32, mults=(1, 2, 2))`: 28 → 14 → 7 rồi ngược lên với skip connection,
khoảng 1.05 triệu tham số. Đọc nó để biết `emb` từ đâu tới.

In [ ]:
def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """(B,) → (B, dim): [sin(t ω_i) | cos(t ω_i)], ω_i = 10000^(-i / (dim / 2))."""
    # TODO
    raise NotImplementedError


class ResBlock(nn.Module):
    """GroupNorm → SiLU → Conv3x3, cộng chiếu time embedding, GroupNorm → SiLU → Conv3x3, cộng skip."""

    def __init__(self, cin: int, cout: int, tdim: int):
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError

In [ ]:
class UNet(nn.Module):
    """U-Net nhỏ dự đoán nhiễu: forward(x (B, C, 28, 28), t (B,)) → (B, C, 28, 28)."""

    def __init__(self, in_ch: int = 1, ch: int = 32, mults: tuple[int, ...] = (1, 2, 2)):
        super().__init__()
        tdim = 4 * ch
        self.ch = ch
        self.time_mlp = nn.Sequential(nn.Linear(ch, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.stem = nn.Conv2d(in_ch, ch, 3, padding=1)
        self.down, self.up, chans, cur = nn.ModuleList(), nn.ModuleList(), [ch], ch
        for i, m in enumerate(mults):
            self.down.append(ResBlock(cur, ch * m, tdim))
            cur = ch * m
            chans.append(cur)
            if i < len(mults) - 1:
                self.down.append(nn.Conv2d(cur, cur, 3, stride=2, padding=1))
                chans.append(cur)
        self.mid = nn.ModuleList([ResBlock(cur, cur, tdim), ResBlock(cur, cur, tdim)])
        for i, m in reversed(list(enumerate(mults))):
            for _ in range(2):
                self.up.append(ResBlock(cur + chans.pop(), ch * m, tdim))
                cur = ch * m
            if i > 0:
                self.up.append(nn.Upsample(scale_factor=2, mode="nearest"))
                self.up.append(nn.Conv2d(cur, cur, 3, padding=1))
        self.out = nn.Sequential(nn.GroupNorm(8, cur), nn.SiLU(), nn.Conv2d(cur, in_ch, 3, padding=1))

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        emb = self.time_mlp(sinusoidal_embedding(t, self.ch))
        h = self.stem(x)
        hs = [h]
        for layer in self.down:
            h = layer(h, emb) if isinstance(layer, ResBlock) else layer(h)
            hs.append(h)
        for layer in self.mid:
            h = layer(h, emb)
        for layer in self.up:
            h = layer(torch.cat([h, hs.pop()], 1), emb) if isinstance(layer, ResBlock) else layer(h)
        return self.out(h)

## 3. Loss, hai sampler và EMA

Mọi số ngẫu nhiên lấy từ `generator` (một `torch.Generator` trên **CPU**) theo đúng thứ tự dưới đây, sinh trên CPU rồi
`.to(device)` của input/model, để cùng seed cho cùng kết quả trên mọi máy.

- `ddpm_loss(model, x0, schedule)` nhận thêm `generator`: lấy `t = torch.randint(0, T, (B,), generator=generator)` rồi
  `noise = torch.randn(x0.shape, generator=generator)`, tạo $$x_t$$ theo dạng đóng, trả MSE (trung bình mọi phần tử) giữa
  `model(x_t, t)` và `noise`.
- `ddpm_sample(model, shape, schedule, generator, clip=True)`: $$x_{T} \sim \mathcal{N}(0, I)$$ (một lần `randn(shape)`), rồi với
  `t = T-1, ..., 0`: dự đoán nhiễu, suy ra $$\hat x_0$$, nếu `clip` thì kẹp $$\hat x_0$$ vào $$[-1, 1]$$, tính mean của posterior
  $$q(x_{t-1} \mid x_t, \hat x_0)$$, và với `t > 0` cộng $$\sqrt{\tilde\beta_t}\,z$$ (một `randn(shape)` mỗi bước, $$\tilde\beta_t$$ là variance của
  posterior); ở `t = 0` trả mean, không cộng nhiễu. Chạy trong `torch.no_grad()`, gọi `model.eval()` trước.
- `ddim_sample(model, shape, schedule, steps, generator, clip=True)`: DDIM với $$\eta = 0$$ (tất định sau khi lấy $$x_T$$):
  các bước `ts = torch.linspace(0, T - 1, steps).round().long()`, đi từ bước cuối về đầu; mỗi bước từ $$t$$ về
  $$t_{prev}$$ (bước liền trước trong `ts`, hoặc "trước 0" với $$\bar\alpha = 1$$):
  $$x_{t_{prev}} = \sqrt{\bar\alpha_{t_{prev}}}\,\hat x_0 + \sqrt{1 - \bar\alpha_{t_{prev}}}\,\hat\varepsilon$$.
  Khi `clip`, kẹp $$\hat x_0$$ rồi **tính lại** $$\hat\varepsilon$$ từ $$x_t$$ và $$\hat x_0$$ đã kẹp.
- `EMA(model, decay)`: giữ bản sao `shadow` (dict tên → tensor, như `state_dict()`); `update(model)` làm
  $$\theta_{ema} \leftarrow d\,\theta_{ema} + (1 - d)\,\theta$$ cho tham số và buffer kiểu số thực (buffer kiểu nguyên thì chép);
  `copy_to(model)` nạp `shadow` vào `model`.

In [ ]:
def _at(v: torch.Tensor, t: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return v.to(x.device)[t].to(x.dtype).reshape(-1, *([1] * (x.dim() - 1)))


def ddpm_loss(model: nn.Module, x0: torch.Tensor, schedule: dict, generator: torch.Generator) -> torch.Tensor:
    """MSE giữa nhiễu thật và nhiễu model dự đoán tại t ngẫu nhiên đều."""
    # TODO
    raise NotImplementedError


@torch.no_grad()
def ddpm_sample(model: nn.Module, shape: tuple[int, ...], schedule: dict, generator: torch.Generator, clip: bool = True) -> torch.Tensor:
    """Sampler tổ tiên T bước, variance = variance posterior (beta tilde)."""
    # TODO
    raise NotImplementedError


@torch.no_grad()
def ddim_sample(model: nn.Module, shape: tuple[int, ...], schedule: dict, steps: int, generator: torch.Generator, clip: bool = True) -> torch.Tensor:
    """DDIM eta = 0 trên `steps` bước cách đều."""
    # TODO
    raise NotImplementedError


class EMA:
    """Exponential moving average của state_dict (tham số và buffer số thực)."""

    def __init__(self, model: nn.Module, decay: float = 0.999):
        # TODO
        raise NotImplementedError

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        # TODO
        raise NotImplementedError

    def copy_to(self, model: nn.Module) -> None:
        # TODO
        raise NotImplementedError

In [ ]:
# explore: đọc FashionMNIST, chọn cấu hình theo device
X_train, y_train = load_idx_images(FASHION / "FashionMNIST/raw", "train")
X_test, y_test = load_idx_images(FASHION / "FashionMNIST/raw", "t10k")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STEPS, BATCH, N_SAMPLE, SCHEDULES = (8000, 128, 1000, ("linear", "cosine")) if DEVICE == "cuda" else (600, 64, 64, ("linear",))
print(X_train.shape, X_test.shape, DEVICE, "| bước train", STEPS, "| batch", BATCH, "| số mẫu sinh", N_SAMPLE)

In [ ]:
# explore: forward process: sqrt(alpha_bar) (phần tín hiệu còn lại) ở vài bước, hai schedule; và ảnh x_t của một áo
import matplotlib.pyplot as plt

sched = {k: make_schedule(1000, k) for k in ("linear", "cosine")}
show_t = [0, 50, 100, 250, 500, 750, 999]
for k, s in sched.items():
    print(f"{k:6s} sqrt(alpha_bar):", [round(s["alpha_bars"][t].sqrt().item(), 3) for t in show_t])
g = torch.Generator().manual_seed(0)
noise = torch.randn(1, 1, 28, 28, generator=g)
fig, axes = plt.subplots(2, len(show_t), figsize=(12, 3.6))
for r, (k, s) in enumerate(sched.items()):
    for j, t in enumerate(show_t):
        ab = s["alpha_bars"][t]
        xt = ab.sqrt() * X_train[0:1] + (1 - ab).sqrt() * noise
        axes[r, j].imshow(xt[0, 0], cmap="gray", vmin=-2, vmax=2); axes[r, j].axis("off")
        axes[r, j].set_title(f"{k} t={t}", fontsize=8)
plt.show()

In [ ]:
# explore: classifier CNN nhỏ (2 epoch) làm giám khảo: mẫu "rõ" thì classifier tự tin, mẫu đa dạng thì phủ đủ 10 lớp
torch.manual_seed(0)
clf = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
                    nn.MaxPool2d(2), nn.Flatten(), nn.Linear(64 * 49, 128), nn.ReLU(), nn.Linear(128, 10)).to(DEVICE)
opt = torch.optim.Adam(clf.parameters(), 1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(2):
    perm = torch.randperm(len(X_train), generator=g)
    for i in range(0, len(X_train), 128):
        b = perm[i:i + 128]
        loss = F.cross_entropy(clf(X_train[b].to(DEVICE)), y_train[b].to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
clf.eval()

@torch.no_grad()
def judge(imgs):
    """(tỉ lệ ảnh classifier tự tin > 0.9, số lớp chiếm ≥ 1%, KL(p̂ || đều) nats)."""
    p = torch.cat([clf(imgs[i:i + 500].to(DEVICE)).softmax(1).cpu() for i in range(0, len(imgs), 500)])
    freq = torch.bincount(p.argmax(1), minlength=10).double() / len(p)
    kl = float(torch.where(freq > 0, freq * (freq * 10).log(), torch.zeros_like(freq)).sum())
    return round((p.max(1).values > 0.9).float().mean().item(), 3), int((freq >= 0.01).sum()), round(kl, 3)

with torch.no_grad():
    acc = (torch.cat([clf(X_test[i:i + 1000].to(DEVICE)).argmax(1).cpu() for i in range(0, 10000, 1000)]) == y_test).float().mean().item()
print("classifier test acc:", round(acc, 4), "| giám khảo trên 1 000 ảnh test thật:", judge(X_test[:1000]))

In [ ]:
# explore: train U-Net (khoảng 1.05 triệu tham số) cho từng schedule, AdamW lr 2e-4, EMA 0.999, lật ngang ngẫu nhiên
def train_ddpm(kind, steps=STEPS, batch=BATCH, seed=0):
    torch.manual_seed(seed)
    model = UNet().to(DEVICE)
    ema = EMA(model, 0.999)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
    s = make_schedule(1000, kind)
    g = torch.Generator().manual_seed(seed)
    losses, t0 = [], time.perf_counter()
    model.train()
    for step in range(steps):
        idx = torch.randint(0, len(X_train), (batch,), generator=g)
        x0 = X_train[idx]
        x0 = torch.where(torch.rand(batch, 1, 1, 1, generator=g) < 0.5, x0.flip(3), x0).to(DEVICE)
        loss = ddpm_loss(model, x0, s, g)
        opt.zero_grad(); loss.backward(); opt.step()
        ema.update(model)
        losses.append(loss.item())
    print(f"{kind}: {sum(p.numel() for p in model.parameters()):,} tham số | loss 100 bước đầu {np.mean(losses[:100]):.4f},"
          f" 100 bước cuối {np.mean(losses[-100:]):.4f} | {time.perf_counter() - t0:.0f} s", flush=True)
    ema_model = UNet().to(DEVICE)
    ema.copy_to(ema_model)
    return model, ema_model, losses

trained = {kind: train_ddpm(kind) for kind in SCHEDULES}

In [ ]:
# explore: so sampler trên model linear: DDPM 1000 bước (EMA và không EMA), DDIM 100/50/20/10 bước (EMA)
raw, ema_model, losses = trained["linear"]
s = sched["linear"]
shape = (N_SAMPLE, 1, 28, 28)
grids, rows = {}, []
def sample_and_judge(name, fn):
    t0 = time.perf_counter()
    imgs = fn(torch.Generator().manual_seed(1)).cpu()
    sec = time.perf_counter() - t0
    grids[name] = imgs[:10]
    rows.append((name, *judge(imgs), round(sec, 1)))
    print(f"{name:22s} tự tin > 0.9 {rows[-1][1]:.3f} | số lớp {rows[-1][2]} | KL {rows[-1][3]:.3f} | {sec:.1f} s", flush=True)

sample_and_judge("DDPM 1000, EMA", lambda g: ddpm_sample(ema_model, shape, s, g))
sample_and_judge("DDPM 1000, không EMA", lambda g: ddpm_sample(raw, shape, s, g))
for k in (100, 50, 20, 10):
    sample_and_judge(f"DDIM {k}, EMA", lambda g, k=k: ddim_sample(ema_model, shape, s, k, g))
sample_and_judge("DDIM 50, EMA, không clip", lambda g: ddim_sample(ema_model, shape, s, 50, g, clip=False))
fig, axes = plt.subplots(len(grids), 10, figsize=(10, 1.1 * len(grids)))
for r, (name, imgs) in enumerate(grids.items()):
    for j in range(10):
        axes[r, j].imshow(imgs[j, 0].clamp(-1, 1), cmap="gray"); axes[r, j].axis("off")
    axes[r, 0].set_title(name, fontsize=7, loc="left")
plt.show()

In [ ]:
# explore: linear và cosine cùng ngân sách (chỉ khi có GPU): loss và mẫu DDIM 50 bước, DDPM 1000 bước
if "cosine" in trained:
    for kind in ("linear", "cosine"):
        _, em, ls = trained[kind]
        sk = sched[kind]
        for name, fn in [("DDIM 50", lambda g: ddim_sample(em, shape, sk, 50, g)), ("DDPM 1000", lambda g: ddpm_sample(em, shape, sk, g))]:
            print(f"{kind:6s} {name:9s}: loss 500 bước cuối {np.mean(ls[-500:]):.4f} | giám khảo {judge(fn(torch.Generator().manual_seed(2)).cpu())}", flush=True)
else:
    print("bỏ qua trên CPU: train thêm một model cosine mất gấp đôi thời gian")

In [ ]:
# explore: đối chiếu với diffusers (không chấm): schedule của DDPMScheduler và sampler DDIMScheduler dùng chính model của bạn
from diffusers import DDIMScheduler, DDPMScheduler

ref = DDPMScheduler(num_train_timesteps=1000, beta_schedule="linear")
print("max |alpha_bar của bạn - diffusers| (linear):", (ref.alphas_cumprod - sched["linear"]["alpha_bars"]).abs().max().item())
ref_cos = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2")
print("max |alpha_bar của bạn - diffusers| (cosine):", (ref_cos.alphas_cumprod - sched["cosine"]["alpha_bars"]).abs().max().item())
ddim = DDIMScheduler(num_train_timesteps=1000, beta_schedule="linear", clip_sample=True, set_alpha_to_one=True)
ddim.set_timesteps(50)
print("50 bước của DDIMScheduler (đầu, cuối):", ddim.timesteps[:3].tolist(), "...", ddim.timesteps[-3:].tolist())
with torch.no_grad():
    x = torch.randn(shape, generator=torch.Generator().manual_seed(1)).to(DEVICE)
    for t in ddim.timesteps:
        x = ddim.step(ema_model(x, torch.full((len(x),), int(t), device=DEVICE)), t, x).prev_sample
print("DDIMScheduler 50 bước, giám khảo:", judge(x.cpu()))

## 4. Đọc kết quả

Trước khi làm trắc nghiệm trong app, tự trả lời:

1. Loss giảm từ đâu tới đâu, và vì sao nó không về gần 0 như loss phân loại? Loss thấp hơn có chắc là mẫu đẹp hơn không?
2. DDIM 50 bước so với DDPM 1 000 bước: chất lượng (theo giám khảo) và thời gian thế nào? Xuống 10 bước thì sao?
3. EMA đổi gì? Bỏ kẹp $$\hat x_0$$ thì sao?
4. Schedule cosine giữ lại bao nhiêu tín hiệu ở giữa quá trình so với linear (cell forward process)? Điều đó ảnh hưởng gì?